####1. Requirement
Read data from students_offline.csv file and load into offline_students_raw table.

In [0]:
offline_students_schema = "ID string, FirstName string, LastName string, Address string, Skills string, Contacts string"

offline_students_raw_df = (
    spark.read.format("csv")
        .option("header", "true")
        .option("quote", "\"")
        .option("escape", "\"")
        .schema(offline_students_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/students_offline.csv")
)

#offline_students_raw_df.display()
offline_students_raw_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students_raw")

####2. Analysis Requirement
We want to know country wise student count.

In [0]:
%sql

with offline_students(
  select id, from_json(address,
      """struct<AddressLine1 string,
        AddressLine2 string,
        City string,
        Country string,
        Pin string,
        State string>
      """) as address
  from dev.spark_db.offline_students_raw
)
select address.country, count(*) as count
from offline_students
group by address.country

country,count
India,3
Engaland,1
Northern Ireland,1


####3. Requirement
Prepare an offline_students table which is ready for analysis

Complex Data Types in Spark
1. Struct
2. Array
3. Map

In [0]:
from pyspark.sql.functions import from_json

address_schema = "struct<AddressLine1 string, AddressLine2 string, City string, Country string, Pin string, State string>"
skills_schema = "array<struct<Skill string, YearsOfExperience string>>"
contacts_schema = "map<string, string>"

offline_students_df = (
    offline_students_raw_df.withColumns({
        "address": from_json("address", address_schema),
        "skills": from_json("skills", skills_schema),
        "contacts": from_json("contacts", contacts_schema)
    })
)

offline_students_df.display()
offline_students_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students")

ID,FirstName,LastName,address,skills,contacts
101,Prashant,Pandey,"List(D104 Gopalan Squire, Whitefield, Bangalore, India, 560001, Karnataka)","List(List(Apache Spark, 5), List(Apache Kafka, 6))","Map(email -> xyz@abc.com, phone -> 9823128923)"
102,David,Turner,"List(109 Park Street, Richmond, London, Engaland, EC1A, London)","List(List(Java, 12), List(Spring Boot, 6))",Map(phone -> 9873145698)
103,Katie,Mcloskey,"List(9th Avenue, Dorsy Road, Belfast, Northern Ireland, BT1 1BG, Belfast)","List(List(SQL, 12), List(PL/SQL, 8))",Map(email -> ert89@abc.com)
104,Nasima,Khatun,"List(G105 MG Tower, Bregade Road, Kolkata, India, 7000001, West Bengal)","List(List(Hadoop, 3), List(Apache Spark, 2))","Map(email -> magt23@abc.com, office -> 7896524689)"
105,Pritam,Jain,"List(M206 Richmond Tower, Electronic City, Bangalore, India, 560001, Karnataka)","List(List(Python, 10), List(SQL, 15), List(Apache Spark, 3), List(Databases, 15))","Map(phone -> 6984753281, whatsapp -> 6924587322)"


####4. Requirement
Perform the following analysis
1. What is country wise student count.
2. Find all students with more than 1 years of Spark knowledge
3. Find all students who didn't provide phone or whatsapp

4.1 What is country wise student count.

In [0]:
%sql

select address.Country, count(*) as count
from dev.spark_db.offline_students
group by address.Country

Country,count
India,3
Engaland,1
Northern Ireland,1


4.2 Find all students with more than 1 years of Spark knowledge

In [0]:
%sql

with offline_students_skills(
   select id, FirstName, LastName, explode(skills) as skills
   from dev.spark_db.offline_students
)
select id, firstname, lastname, skills.*
from offline_students_skills
where skills.Skill like "%Spark%" and skills.YearsOfExperience > 1

id,firstname,lastname,Skill,YearsOfExperience
101,Prashant,Pandey,Apache Spark,5
104,Nasima,Khatun,Apache Spark,2
105,Pritam,Jain,Apache Spark,3


4.3 Find all students who didn't provide phone or whatsapp

In [0]:
%sql

select ID, FirstName, LastName, contacts['email']
from dev.spark_db.offline_students
where contacts['phone'] is null and contacts['whatsapp'] is null

ID,FirstName,LastName,contacts[email]
103,Katie,Mcloskey,ert89@abc.com
104,Nasima,Khatun,magt23@abc.com
